# GameTheory 2d — IESDS et rationalisabilité : élaguer un jeu, puis mesurer ce qui reste

**Navigation** : [<< 2-NormalForm (track principal)](GameTheory-02-NormalForm-Python.ipynb) · [2b-Lean-Definitions](GameTheory-02b-Lean-Definitions-Lean.ipynb) | [Index](README.md) | [4-NashEquilibrium >>](GameTheory-04-NashEquilibrium-Python.ipynb)

**Kernel** : `python3` — aucun appel réseau.

## Pourquoi ce carnet existe

La série pose le jeu en forme normale (`GameTheory-02-NormalForm-Python`), enchaîne sur la topologie des jeux 2×2 (`GameTheory-03-Topology2x2-Python`) puis arrive directement à l'équilibre de Nash (`GameTheory-04-NashEquilibrium-Python`). Entre les deux, il manque **le concept de solution qui les relie** : la **rationalisabilité** — et, en pratique, l'**élimination itérée des stratégies strictement dominées** (IESDS).

**Ce que le carnet 02 fait déjà** — on ne le refait pas : il définit la dominance stricte et implémente une IESDS sur un jeu 3×3 (section 6 de ce carnet-là).

**Ce qu'il ne fait pas** — et que ce carnet fait :

1. justifier que le résultat de l'IESDS **ne dépend pas de l'ordre** d'élimination (théorème pour la dominance stricte, **contre-exemple mesuré** pour la dominance faible) ;
2. distinguer l'IESDS (éliminer ce qui est *dominé*) de l'**admissibilité** (éliminer ce qui n'est *jamais meilleure réponse*) ;
3. relier les deux notions de solution par l'inclusion `rationalisable ⊇ Nash`, et exhiber un profil rationalisable qui **n'est pas** un équilibre de Nash.

## Plan

1. Briques : le module partagé `game_theory_utils.py`, et les fonctions de dominance
2. L'IESDS pas-à-pas, avec son journal d'éliminations
3. L'indépendance à l'ordre : mesure exhaustive sur 4096 jeux 2×3, et contre-exemple en dominance faible
4. L'admissibilité et les stratégies rationalisables
5. `Nash ⊆ rationalisable` : la mesure exhaustive, et un exemple de divergence
6. Exercices

## Prérequis

`GameTheory-02-NormalForm-Python` (forme normale, dominance, meilleure réponse) et, pour la version formelle des mêmes notions, `GameTheory-02b-Lean-Definitions-Lean`. L'équilibre de Nash lui-même est repris de `GameTheory-04-NashEquilibrium-Python` : ici on ne fait que le comparer aux ensembles rationalisables.

In [1]:
# Imports et localisation du module partage de la serie
import itertools
import pathlib
import sys

import numpy as np


def _dossier_serie():
    '''Dossier de la serie GameTheory (celui qui contient game_theory_utils.py).

    On cherche depuis le repertoire courant et ses parents : le carnet
    fonctionne ainsi qu'on l'execute depuis la racine du depot ou depuis
    le dossier de la serie.
    '''
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / "game_theory_utils.py").is_file():
            return base
    return None


DOSSIER_SERIE = _dossier_serie()
if DOSSIER_SERIE is None:
    raise FileNotFoundError(
        "game_theory_utils.py introuvable : executer ce carnet depuis "
        "MyIA.AI.Notebooks/GameTheory ou depuis la racine du depot"
    )
if str(DOSSIER_SERIE) not in sys.path:
    sys.path.insert(0, str(DOSSIER_SERIE))

# Briques deja ecrites par la serie : on les utilise, on ne les reecrit pas.
from game_theory_utils import NormalFormGame, find_pure_nash

print("Dossier de la serie :", DOSSIER_SERIE.name)
print("Classe importee    :", NormalFormGame.__module__ + ".NormalFormGame")

Dossier de la serie : GameTheory
Classe importee    : game_theory_utils.NormalFormGame


## 1. La dominance stricte, et ce qu'elle ne dit pas

Une stratégie $s_i$ **domine strictement** une stratégie $s_i'$ si le joueur $i$ fait **strictement mieux** avec $s_i$ qu'avec $s_i'$, **quelle que soit** la stratégie des autres :

$$u_i(s_i, s_{-i}) > u_i(s_i', s_{-i}) \quad \forall s_{-i} \in S_{-i}$$

Deux points qu'il faut garder en tête avant d'en déduire :

- la dominance est **stricte** : si l'inégalité est partout large et égale quelque part, on parle de dominance *faible* — et la section 3 montrera que les deux ne se comportent pas du tout pareil ;
- la dominance se lit **joueur par joueur** : une stratégie de la Ligne ne peut être éliminée que si *la Ligne* est mieux servie par une autre de ses stratégies.

Le module `game_theory_utils.py` fournit la représentation des jeux (`NormalFormGame`) et la recherche des équilibres de Nash purs (`find_pure_nash`). Les fonctions de dominance ci-dessous sont **nouvelles** : le module partagé ne les contient pas.

In [2]:
# Fonctions de dominance stricte, orientees par joueur
def dominance_stricte_lignes(A):
    '''A[m, n] : gains du joueur Ligne (les lignes sont SES strategies).

    Renvoie un dict {ligne -> index de la strategie qui la domine, ou None}.
    Une ligne est strictement dominee si une AUTRE ligne la bat sur
    TOUTES les colonnes.
    '''
    dominateurs = {}
    for i in range(A.shape[0]):
        dominateur = None
        for k in range(A.shape[0]):
            if k != i and np.all(A[k, :] > A[i, :]):
                dominateur = k
                break
        dominateurs[i] = dominateur
    return dominateurs


def dominance_stricte_colonnes(B):
    '''B[m, n] : gains du joueur Colonne (les colonnes sont SES strategies).

    Meme logique que pour les lignes, mais en transposant : pour le joueur
    Colonne, les strategies de l'adversaire sont les lignes.
    '''
    Bt = B.T
    dominateurs = {}
    for j in range(B.shape[1]):
        dominateur = None
        for l in range(B.shape[1]):
            if l != j and np.all(Bt[l, :] > Bt[j, :]):
                dominateur = l
                break
        dominateurs[j] = dominateur
    return dominateurs


# Verification sur un jeu canonique : le Dilemme du Prisonnier
IPD_A = np.array([[3, 0], [5, 1]], dtype=float)
IPD_B = np.array([[3, 5], [0, 1]], dtype=float)

dom_lignes = dominance_stricte_lignes(IPD_A)
dom_colonnes = dominance_stricte_colonnes(IPD_B)
print("Ligne 0 (Cooperer) dominee par :", dom_lignes[0])
print("Ligne 1 (Defaire) dominee par :", dom_lignes[1])
print("Colonne 0 (Cooperer) dominee par :", dom_colonnes[0])
print("Colonne 1 (Defaire) dominee par :", dom_colonnes[1])

Ligne 0 (Cooperer) dominee par : 1
Ligne 1 (Defaire) dominee par : None
Colonne 0 (Cooperer) dominee par : 1
Colonne 1 (Defaire) dominee par : None


### Lecture du résultat

Les quatre lignes affichent `1` ou `None` : `Cooperer` est strictement dominée par `Defaire` (5 > 3 et 1 > 0), chez la Ligne comme chez la Colonne, tandis que `Defaire` n'est dominée par rien. Le Dilemme du Prisonnier a donc, pour chaque joueur, une stratégie strictement dominée (`Cooperer`) et une stratégie indomptable (`Defaire`) — c'est exactement ce que l'IESDS va énumérer, étape par étape.

## 2. L'IESDS pas-à-pas

La **procédure** IESDS est la suivante : on cherche une stratégie strictement dominée, on la retire du jeu, on recommence sur le jeu réduit, jusqu'à ce qu'aucune stratégie ne soit plus dominée.

Deux points de méthode que la procédure cache :

- l'**ordre** dans lequel on retire les stratégies dominées (section 3) ;
- le fait qu'**éliminer une stratégie peut en créer une nouvelle** : sur le jeu réduit, une stratégie qui n'était pas dominée peut le devenir, parce que la stratégie qui la dominait a disparu.

Le code ci-dessous tient un **journal** des éliminations : chaque étape indique le joueur, la stratégie retirée et la stratégie qui la dominait.

In [3]:
# L'IESDS pas-a-pas, avec journal
def iesds(A, B, labels_l=None, labels_c=None, ordre=None):
    '''Elimination iteree des strategies strictement dominees.

    A, B : matrices de gains (m x n) des deux joueurs.
    labels_l, labels_c : noms des strategies (defaut : R0, R1, ... / C0, C1, ...).
    ordre : priorite d'examen, liste de ("L", index) ou ("C", index). Les index
            sont les index ORIGINAUX des strategies : la priorite reste donc
            stable quand le jeu se reduit.

    Renvoie (survivants_lignes, survivants_colonnes, journal, A_reduit, B_reduit).
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    Ared = np.array(A, dtype=float)
    Bred = np.array(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    if ordre is None:
        ordre = [("L", i) for i in range(len(tous_l))] + [("C", j) for j in range(len(tous_c))]

    journal = []
    etape = 0
    while True:
        elimine = False
        for joueur, index in ordre:
            if joueur == "L":
                if index not in actifs_l:
                    continue
                pos = actifs_l.index(index)
                dominante = None
                for k in actifs_l:
                    if k != index and np.all(Ared[actifs_l.index(k), :] > Ared[pos, :]):
                        dominante = k
                        break
                if dominante is not None:
                    etape += 1
                    journal.append(
                        "Etape %d : Ligne elimine %s, strictement dominee par %s"
                        % (etape, tous_l[index], tous_l[dominante]))
                    actifs_l.remove(index)
                    Ared = np.delete(Ared, pos, axis=0)
                    Bred = np.delete(Bred, pos, axis=0)
                    elimine = True
                    break
            else:
                if index not in actifs_c:
                    continue
                pos = actifs_c.index(index)
                dominante = None
                for l in actifs_c:
                    if l != index and np.all(Bred[:, actifs_c.index(l)] > Bred[:, pos]):
                        dominante = l
                        break
                if dominante is not None:
                    etape += 1
                    journal.append(
                        "Etape %d : Colonne elimine %s, strictement dominee par %s"
                        % (etape, tous_c[index], tous_c[dominante]))
                    actifs_c.remove(index)
                    Ared = np.delete(Ared, pos, axis=1)
                    Bred = np.delete(Bred, pos, axis=1)
                    elimine = True
                    break
        if not elimine:
            break
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c],
            journal, Ared, Bred)

### Un jeu en cascade

Le jeu ci-dessous est construit pour que l'élimination soit **en cascade** : une stratégie de la Colonne n'est dominée qu'**après** le retrait d'une stratégie de la Ligne. C'est le point pédagogique central — la dominance se calcule sur le jeu **réduit**, pas sur le jeu initial.

In [4]:
# Jeu en cascade : le retrait de R0 rend C2 strictement dominee
A_cascade = np.array([
    [0, 0, 0],   # R0
    [1, 1, 1],   # R1
    [3, 0, 3],   # R2
], dtype=float)

B_cascade = np.array([
    [0, 0, 0],    # R0
    [2, 1, 1],    # R1
    [0, 3, -1],   # R2
], dtype=float)

jeu_cascade = NormalFormGame(A_cascade, B_cascade,
                              ["R0", "R1", "R2"], ["C0", "C1", "C2"],
                              name="Jeu en cascade")
jeu_cascade.display()

print("Strategies strictement dominees AVANT toute elimination :")
print("  Ligne   :", dominance_stricte_lignes(A_cascade))
print("  Colonne :", dominance_stricte_colonnes(B_cascade))

surv_l, surv_c, journal, A_red, B_red = iesds(A_cascade, B_cascade,
                                              ["R0", "R1", "R2"], ["C0", "C1", "C2"])
print()
print("Journal de l'IESDS :")
for ligne in journal:
    print("  " + ligne)
print()
print("Survivants : lignes", surv_l, "| colonnes", surv_c)
print("Jeu reduit final :")
print(A_red)
print(B_red)


Jeu en cascade
                    C0            C1            C2
--------------------------------------------------
      R0  ( 0.0, 0.0)  ( 0.0, 0.0)  ( 0.0, 0.0)  
      R1  ( 1.0, 2.0)  ( 1.0, 1.0)  ( 1.0, 1.0)  
      R2  ( 3.0, 0.0)  ( 0.0, 3.0)  ( 3.0,-1.0)  
Strategies strictement dominees AVANT toute elimination :
  Ligne   : {0: 1, 1: None, 2: None}
  Colonne : {0: None, 1: None, 2: None}

Journal de l'IESDS :
  Etape 1 : Ligne elimine R0, strictement dominee par R1
  Etape 2 : Colonne elimine C2, strictement dominee par C0

Survivants : lignes ['R1', 'R2'] | colonnes ['C0', 'C1']
Jeu reduit final :
[[1. 1.]
 [3. 0.]]
[[2. 1.]
 [0. 3.]]


### Lecture du résultat

Avant toute élimination, **seule** `R0` est strictement dominée (par `R1`, qui vaut `1 > 0` sur les trois colonnes) ; aucune stratégie de la Colonne ne l'est encore — `C2` vaut `[0, 1, -1]` et `C0` vaut `[0, 2, 0]` : sur la ligne `R0`, les deux sont à égalité (`0`), donc `C0` ne domine **pas** strictement `C2`.

Le journal montre la cascade en deux temps :

1. **R0 est éliminée** par `R1` ;
2. sur le jeu 2×3 restant, `C2` devient strictement dominée par `C0` (`2 > 1` sur `R1`, `0 > -1` sur `R2`) : elle est éliminée à son tour.

Le jeu final est un 2×2 (`R1`, `R2` × `C0`, `C1`). Deux remarques :

- l'**ordre** des deux éliminations n'a pas d'importance ici (elles sont de joueurs différents) — mais qu'en est-il en général ? C'est la question de la section suivante ;
- le jeu réduit **ne dit pas** quel sera l'équilibre : élaguer un jeu, ce n'est pas le résoudre.

## 3. L'indépendance à l'ordre

**Énoncé.** Pour la dominance **stricte**, l'IESDS est *indépendante de l'ordre* : quelle que soit la stratégie dominée retirée en premier, l'ensemble final des survivants est le même.

L'idée de la preuve est simple : la dominance stricte est une relation « pour tout » (sur toutes les stratégies de l'adversaire), donc elle survit au rétrécissement du jeu ; et la relation d'ordre sur les dominateurs est **stricte** (un dominateur strict ne peut pas être lui-même strictement dominé par une stratégie dominée, sans cycle de gains). Le résultat est donc un point fixe, indépendant du chemin.

**Faut-il le croire ?** Non : on le mesure. Le code ci-dessous parcourt **tous** les jeux 2×3 dont les gains sont dans `{0, 1}` — soit $2^6 \times 2^6 = 4096$ jeux — et, pour chacun, compare les survivants obtenus sous plusieurs ordres d'élimination tirés au hasard (graine fixée, donc reproductible).

In [5]:
# Mesure exhaustive : l'IESDS stricte depend-elle de l'ordre ?
def tous_les_jeux_2x3():
    '''Tous les jeux 2x3 dont les gains des deux joueurs sont dans {0, 1}.'''
    for a_flat in itertools.product((0, 1), repeat=6):
        A = np.array(a_flat, dtype=float).reshape(2, 3)
        for b_flat in itertools.product((0, 1), repeat=6):
            B = np.array(b_flat, dtype=float).reshape(2, 3)
            yield A, B


def ordres_aleatoires(rng, m, n, nombre):
    '''Ordres d'examen aleatoires des strategies (lignes et colonnes melangees).'''
    elements = [("L", i) for i in range(m)] + [("C", j) for j in range(n)]
    for _ in range(nombre):
        melange = list(elements)
        rng.shuffle(melange)
        yield melange


rng = np.random.default_rng(0)
nombre_jeux = 0
divergences = []
for A, B in tous_les_jeux_2x3():
    nombre_jeux += 1
    survivants = set()
    for ordre in ordres_aleatoires(rng, 2, 3, 10):
        lig, col, _, _, _ = iesds(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ordre)
        survivants.add((tuple(lig), tuple(col)))
    if len(survivants) > 1:
        divergences.append((A.copy(), B.copy(), sorted(survivants)))

print("Jeux 2x3 a gains dans {0, 1} analyses : %d" % nombre_jeux)
print("Jeux dont les survivants dependent de l'ordre : %d" % len(divergences))
print("Ordres tires par jeu : 10 (graine 0, reproductible)")

Jeux 2x3 a gains dans {0, 1} analyses : 4096
Jeux dont les survivants dependent de l'ordre : 0
Ordres tires par jeu : 10 (graine 0, reproductible)


### Lecture du résultat

Sur les **4096** jeux 2×3 exhaustivement énumérés, avec dix ordres d'élimination tirés au hasard par jeu, **aucun** jeu ne présente de divergence : l'ensemble des survivants est indépendant de l'ordre. C'est la mesure, pas l'intuition, qui l'établit — et elle est reproductible (graine `0`).

Cette propriété est **spécifique à la dominance stricte**. Avec la dominance *faible* (inégalité large partout, stricte quelque part), l'ordre change le résultat : on le cherche ci-dessous.

### La dominance faible : un ordre qui change tout

Une stratégie $s_i$ est **faiblement dominée** par $s_i'$ si $u_i(s_i', s_{-i}) \geq u_i(s_i, s_{-i})$ pour tout $s_{-i}$, avec inégalité stricte quelque part. La différence avec le cas strict est décisive : la stratégie faiblement dominée peut être **elle-même** la dominatrice d'une autre stratégie. Supprimer la dominée peut donc supprimer sa dominatrice, et l'ordre devient significatif.

Plutôt que d'inventer un exemple à la main, on **cherche** un contre-exemple dans la même famille exhaustive — 4096 jeux 2×3 à gains dans `{0, 1}` — en comparant deux ordres opposés.

In [6]:
# Contre-exemple : l'elimination en dominance FAIBLE depend de l'ordre
def iesds_faible(A, B, labels_l=None, labels_c=None, ordre=None):
    '''Elimination iteree des strategies FAIBLEMENT dominees.

    Une strategie est faiblement dominee s'il existe une autre strategie
    qui fait au moins aussi bien partout, et strictement mieux quelque part.
    Les index de `ordre` sont les index ORIGINAUX des strategies.
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    Ared = np.array(A, dtype=float)
    Bred = np.array(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    if ordre is None:
        ordre = [("L", i) for i in range(len(tous_l))] + [("C", j) for j in range(len(tous_c))]

    journal = []
    etape = 0
    while True:
        elimine = False
        for joueur, index in ordre:
            if joueur == "L":
                if index not in actifs_l:
                    continue
                pos = actifs_l.index(index)
                dominante = None
                for k in actifs_l:
                    if (k != index
                            and np.all(Ared[actifs_l.index(k), :] >= Ared[pos, :])
                            and np.any(Ared[actifs_l.index(k), :] > Ared[pos, :])):
                        dominante = k
                        break
                if dominante is not None:
                    etape += 1
                    journal.append("Etape %d : Ligne elimine %s (faiblement dominee par %s)"
                                   % (etape, tous_l[index], tous_l[dominante]))
                    actifs_l.remove(index)
                    Ared = np.delete(Ared, pos, axis=0)
                    Bred = np.delete(Bred, pos, axis=0)
                    elimine = True
                    break
            else:
                if index not in actifs_c:
                    continue
                pos = actifs_c.index(index)
                dominante = None
                for l in actifs_c:
                    if (l != index
                            and np.all(Bred[:, actifs_c.index(l)] >= Bred[:, pos])
                            and np.any(Bred[:, actifs_c.index(l)] > Bred[:, pos])):
                        dominante = l
                        break
                if dominante is not None:
                    etape += 1
                    journal.append("Etape %d : Colonne elimine %s (faiblement dominee par %s)"
                                   % (etape, tous_c[index], tous_c[dominante]))
                    actifs_c.remove(index)
                    Ared = np.delete(Ared, pos, axis=1)
                    Bred = np.delete(Bred, pos, axis=1)
                    elimine = True
                    break
        if not elimine:
            break
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c], journal)


ORDRE_A = [("L", 0), ("L", 1), ("C", 0), ("C", 1), ("C", 2)]
ORDRE_B = list(reversed(ORDRE_A))

contre_exemples = []
for A, B in tous_les_jeux_2x3():
    lig_a, col_a, _ = iesds_faible(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ORDRE_A)
    lig_b, col_b, _ = iesds_faible(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ORDRE_B)
    if (lig_a, col_a) != (lig_b, col_b):
        contre_exemples.append((A.copy(), B.copy(), lig_a, col_a, lig_b, col_b))

print("Jeux 2x3 ou l'ordre change le resultat (dominance faible) : %d" % len(contre_exemples))

if contre_exemples:
    A_ex, B_ex, lig_a, col_a, lig_b, col_b = contre_exemples[0]
    print()
    print("Contre-exemple (premier trouve) :")
    print("  Ordre A : L0, L1, C0, C1, C2  ->  lignes", lig_a, "| colonnes", col_a)
    print("  Ordre B : l'ordre inverse       ->  lignes", lig_b, "| colonnes", col_b)
    print()
    print("  Gains de la Ligne (lignes R0, R1) :")
    for i, ligne in enumerate(A_ex):
        print("   R%d" % i, ligne)
    print("  Gains de la Colonne (colonnes C0, C1, C2) :")
    for j in range(B_ex.shape[1]):
        print("   C%d" % j, B_ex[:, j])

Jeux 2x3 ou l'ordre change le resultat (dominance faible) : 1536

Contre-exemple (premier trouve) :
  Ordre A : L0, L1, C0, C1, C2  ->  lignes ['R1'] | colonnes ['C1']
  Ordre B : l'ordre inverse       ->  lignes ['R0', 'R1'] | colonnes ['C1']

  Gains de la Ligne (lignes R0, R1) :
   R0 [0. 0. 0.]
   R1 [0. 0. 1.]
  Gains de la Colonne (colonnes C0, C1, C2) :
   C0 [0. 0.]
   C1 [0. 1.]
   C2 [0. 0.]


### Lecture du résultat

Le compteur **non nul** est le résultat : dans la famille exhaustive des 4096 jeux 2×3 à gains dans `{0, 1}`, il existe des jeux dont l'élimination en dominance faible **dépend de l'ordre**. Le jeu affiché le montre concrètement : les deux ordres ne laissent pas les mêmes stratégies.

La conclusion est donc double, et elle est la même des deux côtés :

- **dominance stricte** : l'ordre n'a aucune importance (mesuré sur 4096 jeux, section précédente) ;
- **dominance faible** : l'ordre a de l'importance (contre-exemple mesuré, ci-dessus).

C'est la raison pour laquelle l'IESDS — et non l'élimination en dominance faible — est la procédure canonique associée aux concepts de solution de la série : elle est **confluente**, donc elle définit un objet (le jeu réduit) et non une procédure.

## 4. L'admissibilité : les stratégies rationalisables

L'IESDS élimine ce qui est *dominé*. Une autre notion, plus faible et plus exigeante, élimine ce qui n'est **jamais une meilleure réponse**.

Une stratégie de la Ligne est **admissible** si elle est la meilleure réponse (à égalité près) à *au moins une* stratégie de la Colonne qui survit au processus. On élimine alors, itérativement, les stratégies qui ne sont la meilleure réponse à **aucune** des stratégies adverses survivantes.

Pourquoi « à au moins une » suffit, et pourquoi il ne faut pas chercher plus : pour un joueur adverse qui joue un **mélange** de ses stratégies survivantes, une stratégie qui n'est meilleure réponse à aucune stratégie pure ne l'est à aucun mélange non plus. En effet, si pour chaque stratégie pure $j$ il existe $k_j$ tel que $u(k_j, j) > u(i, j)$, alors, pour toute distribution $p$ sur les $j$, le **mélange** $q = \sum_j p_j \, k_j$ des meilleures réponses à chaque $j$ vérifie :

$$u(q, p) \;=\; \sum_j p_j \, u(k_j, j) \;>\; \sum_j p_j \, u(i, j) \;=\; u(i, p)$$

L'inégalité est **stricte terme à terme** : pour chaque $j$ de probabilité non nulle, $k_j$ bat $i$ sur $j$. Le mélange $q$ est donc une meilleure réponse stricte à $p$, et la stratégie $i$ n'en est **jamais** une.

C'est l'ensemble des stratégies ainsi survivantes que l'on appelle **rationalisables**, et l'ensemble des profils qu'elles forment : les **profils rationalisables**.

In [7]:
# Admissibilite : elimination iteree des strategies jamais meilleure reponse
def strategies_admissibles(A, B, labels_l=None, labels_c=None):
    '''Elimination iteree des strategies jamais meilleure reponse.

    A chaque tour, on compare les strategies SURVIVANTES (jeu restreint) :
    une strategie est retiree si elle n'est la meilleure reponse a aucune
    des strategies adverses survivantes.

    Renvoie (lignes_admissibles, colonnes_admissibles).
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    while True:
        # Un joueur sans strategie ne laisse plus rien a comparer : on s'arrete
        # plutot que de vider la liste adverse (le max() n'aurait plus d'argument).
        if not actifs_l or not actifs_c:
            break
        Ar = A[np.ix_(actifs_l, actifs_c)]
        Br = B[np.ix_(actifs_l, actifs_c)]

        def ligne_est_meilleure_reponse(i, j):
            '''La ligne i est-elle meilleure reponse a la colonne j ?'''
            meilleur = max(Ar[p, j] for p in range(len(actifs_l)))
            return Ar[i, j] >= meilleur

        def colonne_est_meilleure_reponse(j, i):
            '''La colonne j est-elle meilleure reponse a la ligne i ?'''
            meilleur = max(Br[i, q] for q in range(len(actifs_c)))
            return Br[i, j] >= meilleur

        a_retirer_l = [p for p in range(len(actifs_l))
                       if not any(ligne_est_meilleure_reponse(p, q)
                                  for q in range(len(actifs_c)))]
        a_retirer_c = [q for q in range(len(actifs_c))
                       if not any(colonne_est_meilleure_reponse(q, p)
                                  for p in range(len(actifs_l)))]
        if not a_retirer_l and not a_retirer_c:
            break
        for p in sorted(a_retirer_l, reverse=True):
            actifs_l.pop(p)
        for q in sorted(a_retirer_c, reverse=True):
            actifs_c.pop(q)
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c])


# La Bataille des Sexes : le classique de la divergence
A_BOS = np.array([[2, 0], [0, 1]], dtype=float)   # Op, Ba
B_BOS = np.array([[1, 0], [0, 2]], dtype=float)   # Op, Ba
jeu_bos = NormalFormGame(A_BOS, B_BOS, ["Opera", "Ballet"], ["Opera", "Ballet"],
                         name="Bataille des Sexes")
jeu_bos.display()

nash_purs = find_pure_nash(jeu_bos)
adm_l, adm_c = strategies_admissibles(A_BOS, B_BOS, ["Opera", "Ballet"], ["Opera", "Ballet"])

print("Equilibres de Nash purs            :", [(jeu_bos.row_labels[i], jeu_bos.col_labels[j])
                                                 for i, j in nash_purs])
print("Strategies rationalisables (Ligne) :", adm_l)
print("Strategies rationalisables (Colonne):", adm_c)

profils_rationalisables = [(l, c) for l in adm_l for c in adm_c]
profils_nash = [(jeu_bos.row_labels[i], jeu_bos.col_labels[j]) for i, j in nash_purs]
divergents = [p for p in profils_rationalisables if p not in profils_nash]

print()
print("Profils rationalisables : %d" % len(profils_rationalisables))
print("Profils de Nash          : %d" % len(profils_nash))
print("Rationalisables mais PAS de Nash :", divergents)


Bataille des Sexes
                 Opera        Ballet
------------------------------------
   Opera  ( 2.0, 1.0)  ( 0.0, 0.0)  
  Ballet  ( 0.0, 0.0)  ( 1.0, 2.0)  
Equilibres de Nash purs            : [('Opera', 'Opera'), ('Ballet', 'Ballet')]
Strategies rationalisables (Ligne) : ['Opera', 'Ballet']
Strategies rationalisables (Colonne): ['Opera', 'Ballet']

Profils rationalisables : 4
Profils de Nash          : 2
Rationalisables mais PAS de Nash : [('Opera', 'Ballet'), ('Ballet', 'Opera')]


### Lecture du résultat

Dans la Bataille des Sexes, les **deux** stratégies de chaque joueur sont rationalisables (`Opera` et `Ballet` pour chacun : chacune est meilleure réponse à la stratégie correspondante de l'adversaire), alors que seuls **deux** profils sont des équilibres de Nash : `(Opera, Opera)` et `(Ballet, Ballet)`.

Le profil `(Opera, Ballet)` est donc **rationalisable mais pas de Nash** — c'est la divergence recherchée. Elle a une lecture immédiate : après l'élimination, chaque joueur sait que l'adversaire peut jouer `Opera` ou `Ballet` ; mais si les deux jouent `Opera`/`Ballet`, l'un d'eux ferait mieux de changer. Rationalisable dit « je n'exclurai pas ce profil », Nash dit « personne n'y dévie ».

## 5. `Nash ⊆ rationalisable` — la mesure exhaustive

L'inclusion se comprend : dans un équilibre de Nash, la stratégie de chaque joueur est une meilleure réponse à celle des autres ; elle n'est donc **jamais** éliminée par le processus d'admissibilité, qui ne retire que les stratégies qui ne sont meilleure réponse à rien.

Mais cette inclusion a un sens **plus fort** qu'elle n'en a l'air : elle doit survivre à toutes les éliminations, y compris celles rendues possibles par les éliminations précédentes. On la vérifie donc sur la même famille exhaustive de 4096 jeux : pour chaque jeu, on compare les équilibres de Nash purs aux profils rationalisables.

On compare au passage les **deux filtres** côte à côte : l'IESDS (éliminer ce qui est *dominé*) et l'admissibilité (éliminer ce qui n'est *jamais meilleure réponse*). Une stratégie strictement dominée n'est jamais une meilleure réponse : elle est donc éliminée par l'admissibilité **dès le premier tour**. L'inclusion attendue est donc `Rationalisable ⊆ survivants de l'IESDS` — et l'on mesure de combien elle est stricte, car les deux filtres ne répondent pas à la même question.

In [8]:
# Verification exhaustive : Nash <= rationalisable, et comparaison des deux filtres
violations_nash = []
violations_inclusion = 0
divergences_iesds = 0
profils_nash_total = 0
profils_rationalisables_total = 0
jeux_avec_nash = 0

for A, B in tous_les_jeux_2x3():
    jeu = NormalFormGame(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    nash = find_pure_nash(jeu)
    adm_l, adm_c = strategies_admissibles(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    rationalisables = set((l, c) for l in adm_l for c in adm_c)

    profils_nash_total += len(nash)
    profils_rationalisables_total += len(rationalisables)
    if nash:
        jeux_avec_nash += 1
    for i, j in nash:
        if ("R%d" % i, "C%d" % j) not in rationalisables:
            violations_nash.append((A.copy(), B.copy(), (i, j)))

    # Comparaison des deux filtres : l'admissibilite est-elle incluse dans l'IESDS ?
    iesds_l, iesds_c, _, _, _ = iesds(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    if not set(adm_l).issubset(set(iesds_l)) or not set(adm_c).issubset(set(iesds_c)):
        violations_inclusion += 1
    # Et l'inverse : l'IESDS garde-t-elle des strategies que l'admissibilite elimine ?
    if not set(iesds_l).issubset(set(adm_l)) or not set(iesds_c).issubset(set(adm_c)):
        divergences_iesds += 1

print("Jeux analyses                               : %d" % nombre_jeux)
print("Jeux avec au moins un equilibre de Nash     : %d" % jeux_avec_nash)
print("Profils de Nash rencontres au total        : %d" % profils_nash_total)
print("Profils rationalisables rencontres         : %d" % profils_rationalisables_total)
print("Contre-exemples a Nash <= rationalisable   : %d" % len(violations_nash))
print("Contre-exemples a Rationalisable <= IESDS   : %d" % violations_inclusion)
print("Jeux ou l'IESDS garde plus que l'admissibilite : %d" % divergences_iesds)

Jeux analyses                               : 4096
Jeux avec au moins un equilibre de Nash     : 4066
Profils de Nash rencontres au total        : 11520
Profils rationalisables rencontres         : 20286
Contre-exemples a Nash <= rationalisable   : 0
Contre-exemples a Rationalisable <= IESDS   : 0
Jeux ou l'IESDS garde plus que l'admissibilite : 372


### Lecture du résultat

Trois compteurs, trois faits distincts.

**1. `Nash ⊆ rationalisable` tient.** Le compteur de contre-exemples est **zéro** : sur les 4096 jeux exhaustivement énumérés, chaque profil de Nash est bien un profil rationalisable. Ce n'est donc pas une intuition de manuel — elle tient sur toute la famille.

**2. La divergence est de taille.** 20286 profils rationalisables contre 11520 profils de Nash sur la même famille : la rationalisabilité est strictement plus permissive. Autrement dit, la rationalisabilité est une **approximation** de l'équilibre de Nash — plus facile à vérifier, plus robuste, mais elle ne dit pas quel sera l'équilibre. C'est le sens de l'inclusion :

$$\text{Nash} \subsetneq \text{Rationalisable}$$

**3. Les deux filtres ne sont pas le même filtre — et l'inclusion va dans le sens inverse de ce qu'on attendait.** Le compteur `Rationalisable <= IESDS` est **zéro** : l'admissibilité ne conserve jamais une stratégie que l'IESDS aurait éliminée (une stratégie strictement dominée n'est jamais une meilleure réponse, elle tombe donc dès le premier tour). Mais le dernier compteur est **non nul** : dans 372 jeux sur 4096, l'admissibilité élimine une stratégie que la dominance stricte avait conservée.

Autrement dit, **l'IESDS est plus grossière que l'admissibilité**. Les deux procédures ne sont pas emboîtées l'une dans l'autre : l'IESDS répond à « qui est *dominé* ? », l'admissibilité à « qui n'est *jamais optimal* ? ». Une stratégie peut être non dominée et néanmoins ne jamais être une meilleure réponse — l'admissibilité la retire, l'IESDS la garde.

C'est la raison mathématique de l'inclusion `Nash ⊆ rationalisable` : seul le filtre par optimalité est **garanti** de conserver tous les équilibres, pas le filtre par dominance.

## 6. Exercices

Quatre exercices pour mettre en pratique les outils du carnet. Les solutions ne sont pas fournies : chaque cellule de code est un squelette à compléter (`# TODO`). Toutes les briques nécessaires (`NormalFormGame`, `find_pure_nash`, `dominance_stricte_lignes`, `dominance_stricte_colonnes`, `iesds`, `iesds_faible`, `strategies_admissibles`, `tous_les_jeux_2x3`) restent disponibles dans l'environnement du carnet.

### Exercice 1 : l'IESDS pas-à-pas sur un jeu 3×3

Reprenez le jeu en cascade du carnet, mais avec des gains **différents** : la dynamique doit être la même — une stratégie de la Colonne ne devient dominée qu'après le retrait d'une stratégie de la Ligne.

| | C0 | C1 | C2 |
|---|---|---|---|
| **R0** | (0, 0) | (0, 0) | (0, 0) |
| **R1** | (2, 3) | (2, 2) | (2, 2) |
| **R2** | (4, 0) | (1, 4) | (4, -2) |

**Objectif** : écrire la fonction `iesds_3x3` qui élimine les stratégies strictement dominées une par une et renvoie le **journal** des éliminations.

- **Indice** : `dominance_stricte_lignes` et `dominance_stricte_colonnes` opèrent sur une sous-matrice — il faut les rejouer sur le jeu réduit, à chaque étape.
- **Etape 1** : trouver la première stratégie strictement dominée (les lignes d'abord).
- **Etape 2** : la retirer, puis recommencer sur le jeu réduit.
- **Etape 3** : renvoyer `(survivants_lignes, survivants_colonnes, journal)` et l'afficher.

In [9]:
# Exercice 1 : l'IESDS pas-a-pas sur un jeu 3x3
# ==================================================================
# TODO : completer `iesds_3x3` (elimination + journal).
# TODO etudiant : renvoyer (survivants_lignes, survivants_colonnes, journal)
resultat_ex1 = None  # TODO etudiant

print("Exercice 1 a completer : la cascade attendue comporte deux eliminations.")

Exercice 1 a completer : la cascade attendue comporte deux eliminations.


### Exercice 2 : l'IESDS dépend-elle de l'ordre ?

Reprenez le jeu en cascade du carnet (`A_cascade`, `B_cascade`) et vérifiez **explicitement** que les survivants ne dépendent pas de l'ordre d'élimination.

**Objectif** : comparer les survivants obtenus sous deux ordres différents, et vérifier qu'ils coïncident.

- **Indice** : la fonction `iesds` accepte un paramètre `ordre`, liste de couples `("L", index)` ou `("C", index)`.
- **Etape 1** : construire un ordre où la Colonne est examinée avant la Ligne, et un autre où la Ligne est examinée avant la Colonne.
- **Etape 2** : comparer les deux résultats et afficher un message explicite si (et seulement si) ils diffèrent.
- **Etape 3** : faire de même avec `iesds_faible` (dominance **faible**) sur le même jeu, et dire si la conclusion change.

In [10]:
# Exercice 2 : l'IESDS depend-elle de l'ordre ?
# ==================================================================
# TODO : comparer les survivants de `iesds` sous deux ordres opposes.
# TODO etudiant : afficher "ordre independant" ou "ordre dependant"
conclusion_ex2 = None  # TODO etudiant

print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 : stratégies rationalisables

Considérez le jeu suivant (gains de la Ligne, gains de la Colonne) :

| | C0 | C1 |
|---|---|---|
| **R0** | (2, 0) | (0, 1) |
| **R1** | (1, 0) | (1, 0) |

**Objectif** : calculer les stratégies **rationalisables** de ce jeu et les comparer à ses équilibres de Nash.

- **Indice** : `strategies_admissibles(A, B, labels_l, labels_c)` renvoie les survivants de l'admissibilité ; `find_pure_nash(NormalFormGame(A, B, ...))` renvoie les équilibres purs.
- **Etape 1** : calculer les deux ensembles.
- **Etape 2** : vérifier que tout profil de Nash est rationalisable.
- **Etape 3** : afficher les profils rationalisables qui ne sont pas des équilibres de Nash (il y en a au moins un).

In [11]:
# Exercice 3 : strategies rationalisables
# ==================================================================
A_ex3 = np.array([[2, 0], [1, 1]], dtype=float)
B_ex3 = np.array([[0, 1], [0, 0]], dtype=float)

# TODO : calculer les strategies rationalisables (admissibles) et les
# equilibres de Nash purs, puis comparer les deux ensembles.
# TODO etudiant : afficher les profils rationalisables qui ne sont PAS de Nash
divergence_ex3 = None  # TODO etudiant

print("Exercice 3 a completer")

Exercice 3 a completer

### Exercice 4 : chercher un contre-exemple en dominance faible

La dominance faible rend l'élimination dépendante de l'ordre (mesuré dans le carnet). Votre travail est de **retrouver** un contre-exemple par vous-même, dans une famille que vous choisissez.

**Objectif** : écrire une fonction `chercher_contre_exemple_faible` qui parcourt une famille de jeux et renvoie le premier jeu dont les survivants diffèrent selon l'ordre.

- **Indice** : `iesds_faible` accepte lui aussi un paramètre `ordre` ; `tous_les_jeux_2x3` fournit une famille exhaustive, mais vous pouvez la réduire (par exemple en fixant `A` et en balayant `B`).
- **Etape 1** : choisir la famille à explorer.
- **Etape 2** : comparer les survivants sous deux ordres opposés.
- **Etape 3** : renvoyer `(A, B, survivants_ordre_1, survivants_ordre_2)` et afficher le jeu trouvé.

In [12]:
# Exercice 4 : chercher un contre-exemple en dominance faible
# ==================================================================
def chercher_contre_exemple_faible(famille=None):
    '''Renvoie (A, B, survivants_ordre_1, survivants_ordre_2) ou None.'''
    # TODO etudiant
    return None


contre_exemple_ex4 = None  # TODO etudiant

print("Exercice 4 a completer")

Exercice 4 a completer


## 7. Résumé

| Concept | Définition | Propriété mesurée |
|---|---|---|
| **Dominance stricte** | $u_i(s_i, s_{-i}) > u_i(s_i', s_{-i})$ pour tout $s_{-i}$ | — |
| **IESDS** | élimination itérée des stratégies strictement dominées | **indépendante de l'ordre** (4096 jeux 2×3, 10 ordres chacun) ; garde **plus** de stratégies que l'admissibilité (372 jeux sur 4096) |
| **Dominance faible** | $\geq$ partout, $>$ quelque part | **dépendante de l'ordre** (contre-exemple mesuré) |
| **Admissibilité** | la stratégie est meilleure réponse à au moins une stratégie survivante | ne élimine jamais un équilibre de Nash ; filtre **plus fin** que l'IESDS |
| **Profil rationalisable** | produit de stratégies admissibles | **strictement plus nombreux** que les profils de Nash |

Les deux inclusions à retenir :

$$\text{Nash} \subsetneq \text{Rationalisable} \subseteq \text{IESDS (survivants)}$$

`Nash ⊊ Rationalisable` : 20286 profils rationalisables contre 11520 profils de Nash sur la famille exhaustive, et le profil `(Opera, Ballet)` de la Bataille des Sexes est rationalisable sans être un équilibre.

`Rationalisable ⊆ IESDS` : une stratégie strictement dominée n'est jamais une meilleure réponse, elle est donc éliminée par l'admissibilité dès le premier tour. L'inclusion est **stricte** dans 372 jeux sur 4096 — l'admissibilité y retire une stratégie que la dominance stricte avait conservée. Les deux filtres ne sont donc pas emboîtés : l'IESDS répond à « qui est dominé ? », l'admissibilité à « qui n'est jamais optimal ? ».

## Limites

- La mesure exhaustive porte sur les jeux **2×3 à gains dans `{0, 1}`** : c'est une famille complète pour la *structure* des questions posées (ordre, inclusion), mais pas un quelconque jeu 3×3 ou n×m.
- L'admissibilité est ici définie par élimination des stratégies **jamais meilleure réponse à une stratégie pure adverse** ; pour deux joueurs, c'est équivalent à l'admissibilité au sens des beliefs mélangées (démonstration en section 4). L'admissibilité **au sens strict** (non-domination par un mixte) est un ensemble plus grand que la rationalisabilité : elle n'est pas calculée ici.
- Les deux filtres (dominance et admissibilité) sont comparés **sur la même famille** ; la strictesse de `Rationalisable ⊊ IESDS` (372 jeux sur 4096) est une mesure de cette famille, pas une propriété universelle.
- Les profils **mixtes** ne sont pas traités ici : l'inclusion `Nash ⊆ rationalisable` est vérifiée sur les équilibres **purs**.

## Pour aller plus loin

- [`GameTheory-02-NormalForm-Python`](GameTheory-02-NormalForm-Python.ipynb) — la forme normale, la dominance, l'IESDS et les équilibres purs
- [`GameTheory-02b-Lean-Definitions-Lean`](GameTheory-02b-Lean-Definitions-Lean.ipynb) — la formalisation Lean des mêmes notions (jeu, stratégie, meilleure réponse, Nash)
- [`GameTheory-04-NashEquilibrium-Python`](GameTheory-04-NashEquilibrium-Python.ipynb) — l'équilibre de Nash en détail, y compris mixte

***

**Notebook précédent** : [GameTheory-02-NormalForm-Python](GameTheory-02-NormalForm-Python.ipynb) · [GameTheory-02b-Lean-Definitions-Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb)
**Notebook suivant** : [GameTheory-04-NashEquilibrium-Python](GameTheory-04-NashEquilibrium-Python.ipynb)